In [44]:
# import nltk
# from nltk.corpus import guthenberg
# import pandas as pd


In [45]:
import nltk
from nltk.corpus import gutenberg  # Fixed typo
import pandas as pd

# Download corpus if not already downloaded
nltk.download('gutenberg')

# Load the raw text of Shakespeare's Hamlet
data = gutenberg.raw('shakespeare-hamlet.txt')

# Write the data to a file
with open('dataset.txt', 'w', encoding='utf-8') as f:
    f.write(data)


[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\CC\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [46]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

In [47]:
with open('dataset.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [48]:
Tokenizer = Tokenizer()
Tokenizer.fit_on_texts([text])
total_words = len(Tokenizer.word_index) + 1

In [49]:
input_sequences = []
for line in text.split('\n'):
    token_list = Tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i + 1]
        input_sequences.append(n_gram_sequence)

In [50]:
max_seq_len = max([len(x) for x in input_sequences])
max_seq_len

14

In [51]:
input_seq = np.array(pad_sequences
                     (input_sequences,
                      maxlen=max_seq_len, 
                      padding='pre'))


In [52]:
input_seq

array([[   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       [   0,    0,    0, ...,  687,    4,   45],
       ...,
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4],
       [   0,    0,    0, ..., 1047,    4,  193]])

In [53]:
##create predictors and label
import tensorflow as tf
x,y = input_seq[:,:-1],input_seq[:,-1]

In [54]:
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

In [55]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

In [56]:
## define early stopping
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=3,verbose = 1 ,restore_best_weights=True)

In [57]:
#lstm model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout,GRU
model1 = Sequential()
model1.add(Embedding(total_words, 100, input_length=max_seq_len-1))
model1.add(LSTM(150, return_sequences=True))
model1.add(Dropout(0.2))
model1.add(LSTM(100))
model1.add(Dense(total_words, activation='softmax'))
model1.build(input_shape=(None, max_seq_len-1))
model1.compile(loss = 'categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model1.summary()

c:\Users\CC\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 13, 100)        │       481,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 13, 150)        │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 13, 150)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4818)           │       486,618 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,219,418 (4.65 MB)

 Trainable params: 1,219,418 (4.65 MB)

 Non-trainable params: 0 (0.00 B)

In [58]:
#lstm model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout,GRU
model2 = Sequential()
model2.add(Embedding(total_words, 100, input_length=max_seq_len-1))
model2.add(GRU(150, return_sequences=True))
model2.add(Dropout(0.2))
model2.add(GRU(100))
model2.add(Dense(total_words, activation='softmax'))
model2.build(input_shape=(None, max_seq_len-1))
model2.compile(loss = 'categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model2.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 13, 100)        │       481,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, 13, 150)        │       113,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 13, 150)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ (None, 100)            │        75,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 4818)           │       486,618 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,157,418 (4.42 MB)

 Trainable params: 1,157,418 (4.42 MB)

 Non-trainable params: 0 (0.00 B)

In [59]:
# ## model train by ourself   
# history = model1.fit(x_train, y_train, epochs=100, verbose=1, validation_data=(x_test, y_test), callbacks=[early_stopping])
# # Save the model
# model1.save('lstm_model.h5')
# # Load the model
# from tensorflow.keras.models import load_model
# model1 = load_model('lstm_model.h5')
# # Generate text using the trained model
# def generate_text(seed_text, next_words, model, max_seq_len):
#     for _ in range(next_words):
#         token_list = Tokenizer.texts_to_sequences([seed_text])[0]
#         token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')
#         predicted = model.predict(token_list, verbose=0)
#         predicted_word_index = np.argmax(predicted, axis=-1)[0]
#         output_word = ""
#         for word, index in Tokenizer.word_index.items():
#             if index == predicted_word_index:
#                 output_word = word
#                 break
#         seed_text += " " + output_word
#     return seed_text
# # Example usage
# seed_text = "To be or not to be"
# next_words = 10
# generated_text = generate_text(seed_text, next_words, model1, max_seq_len)
# print(generated_text)


In [60]:
import tensorflow as tf
tf.compat.v1.enable_eager_execution()


In [61]:
history1 = model1.fit(x_train, y_train, epochs=50,callbacks=[early_stopping],validation_data=(x_test, y_test))


Epoch 1/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 26s 33ms/step - accuracy: 0.0293 - loss: 7.1224 - val_accuracy: 0.0334 - val_loss: 6.7491
Epoch 2/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 21s 33ms/step - accuracy: 0.0364 - loss: 6.4676 - val_accuracy: 0.0410 - val_loss: 6.8624
Epoch 3/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.0443 - loss: 6.3117 - val_accuracy: 0.0501 - val_loss: 6.8705
Epoch 4/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 20s 31ms/step - accuracy: 0.0515 - loss: 6.1560 - val_accuracy: 0.0492 - val_loss: 6.8842
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 1.


In [62]:
history2 = model2.fit(x_train, y_train, epochs=50, verbose=1, validation_data=(x_test, y_test), callbacks=[early_stopping])

Epoch 1/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 23s 30ms/step - accuracy: 0.0280 - loss: 7.2260 - val_accuracy: 0.0357 - val_loss: 6.7876
Epoch 2/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.0392 - loss: 6.4590 - val_accuracy: 0.0534 - val_loss: 6.8062
Epoch 3/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 20s 30ms/step - accuracy: 0.0588 - loss: 6.1722 - val_accuracy: 0.0664 - val_loss: 6.7037
Epoch 4/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 18s 28ms/step - accuracy: 0.0734 - loss: 5.8460 - val_accuracy: 0.0717 - val_loss: 6.7635
Epoch 5/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 17s 27ms/step - accuracy: 0.0916 - loss: 5.5262 - val_accuracy: 0.0719 - val_loss: 6.8375
Epoch 6/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 20s 31ms/step - accuracy: 0.1016 - loss: 5.2333 - val_accuracy: 0.0750 - val_loss: 6.9584
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 3.


In [72]:
## function to pridict the next word
def pridict_next_word(model,tokenizzer,text,max_sequence_len):
    token_list = tokenizzer.texts_to_sequences([text])[0]
    if len(token_list) > max_sequence_len:
        token_list = token_list[-(max_sequence_len-1):]
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted = model.predict(token_list)
    predicted_word_index = np.argmax(predicted, axis=-1)
    for word, index in tokenizzer.word_index.items():
        if index == predicted_word_index:
            return word
    return None

In [73]:
input_text = "to be or not to be"
max_seq_len = model1.input_shape[1] + 1
predicted_word = pridict_next_word(model1, Tokenizer, input_text, max_seq_len)
print(f"Predicted next word: {predicted_word}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 336ms/step
Predicted next word: the
